## PDF Extraction and Embedding (Programme Documents)
Method: Docling (Currently used in the Project)

### 1. Extracting raw PDF data

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker 

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import os, re, pickle, copy

llm = ChatOllama(model="llama3.1:8b", validate_model_on_init=True, temperature=0.1) # Changed model to Lllama 3.1 8B.
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

In [2]:
eee_path = "https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/"
pdf_path = "../RAG/ProgramBooklet"

path = os.path.join(pdf_path)
pdfs = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]

# Print the PDF file sizes
for pdf in pdfs:
    file_path = os.path.join(path, pdf)
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"'{pdf}' is {file_size_mb:.2f} MB")

'BEngBSc_Scheme_IAIE_46409_2526.pdf' is 4.70 MB
'BEng_Scheme_EE_46408_2526.pdf' is 4.70 MB
'MSc_EE_46010_2526.pdf' is 2.40 MB
'MSc_EIE_46011_2526.pdf' is 2.15 MB
'MSc_EV_46012_2526.pdf' is 2.22 MB
'MSc_MQ_46013_2526.pdf' is 1.32 MB
'PhDMPhil_EEE_46601_2526.pdf' is 1.63 MB


In [3]:
def regex_enhance(txt):
    text = re.sub(r' {2,}', ' ', txt)  # Strip excess white spaces
    return text

def extract_programme_title(first_page_content, llm_model):
    """Extract programme title from first page using LLM."""
    content_snippet = first_page_content[:2000]
    
    prompt = \
        f"""
        You are extracting the programme title from a university programme booklet's first page.

        *First page content*:
        {content_snippet}

        Extract the full programme title including:
        1. Degree type (e.g., Bachelor of Engineering, Master of Science, PhD)
        2. Discipline (e.g., Electrical Engineering, Electronic and Information Engineering)

        Return only the following:
        1. The programme title with normalized capitalizations
        2. The short form of such title (e.g., BEng / MSc in EE / EIE)
        Do NOT include any additional texts.

        The programme title and its short form in one line, separated by " | ":
        """
    
    # Extract programme title from first page using LLM
    try:
        response = llm_model.invoke(prompt)
        title = response.content.strip() 
        title = re.sub(r'\s+', ' ', title)
        title = title.replace('"', '').replace("'", "")

        if len(title) > 10 and len(title) < 250:
            return title
        else:
            return "Unknown"
    except Exception as e:
        print(f"LLM extraction failed: {e}")
        return "Unknown"

def summarize_table(table_content, context, llm_model):
    prompt = f"""
    The following is a table markdown and its surrounding context from a university programme booklet.
    Summarize the table's data in detail and concisely. It should capture the key information used for vector search in the RAG system.

    =====
    *Context*:
    {context}
    =====
    *Table Content (Markdown)*:
    {table_content}
    =====
    
    Return only a Concise Summary:
    """
    
    response = llm_model.invoke(prompt)
    return response.content.strip()

In [4]:
text_docs = []
table_raw_docs = []
table_summaries = [] # Multi-vector strategy: store summaries linked to original tables

converter = DocumentConverter()
chunker = HybridChunker(merge_peers=True)

for pdf in pdfs:
    full_path = os.path.abspath(f"{pdf_path}/{pdf}")
    print(f"Checking: {full_path}, exists: {os.path.exists(full_path)}")
    
    # Extract programme title from first page
    loader = PyMuPDFLoader(full_path)
    cur_pdf = loader.load()
    programme_title = ""
    if len(cur_pdf) > 0:
        programme_title = extract_programme_title(cur_pdf[0].page_content, llm)
        print(f"PDF: {pdf} -> Programme: {programme_title}")
    
    # Load the doc
    print(f"\nConverting {pdf} into markdown.\n")
    result = converter.convert(full_path)
    doc = result.document
    
    # 1. Process Grouped Texts via Chunker
    for chunk in chunker.chunk(doc):
        # Metadata from the document hierarchy
        metadata = {
            "source": pdf,
            "content_type": "pdf",
            "programme_title": programme_title,
            "page_number": chunk.meta.page_no if chunk.meta and hasattr(chunk.meta, 'page_no') else None,
            "headings": chunk.meta.headings if hasattr(chunk.meta, 'headings') else []
        }
        
        # Only add text docs for substantial text chunks
        if len(chunk.text.strip()) > 20:
            text_doc = Document(
                page_content=chunk.text,
                metadata={**metadata, "type": "text"}
            )
            text_docs.append(text_doc)

    # 2. Process Tables
    elements = list(doc.iterate_items())
    
    # Process elements
    for i, (item, level) in enumerate(elements):
        metadata = {
            "source": pdf,
            "content_type": "pdf",
            "programme_title": programme_title,
            "page_number": item.prov[0].page_no if item.prov else None
        }
        
        # Check if element is a Table using "has attribute or method" function and if its type contains "Table"
        if hasattr(item, 'export_to_markdown') and 'Table' in str(type(item)):
            table_markdown = item.export_to_markdown()
            
            # Get preceding and following context for tables
            prev_item = elements[i-1][0] if i > 0 else ""
            prev_context = prev_item.text[-300:] if hasattr(prev_item, 'text') else "None"
            
            next_item = elements[i+1][0] if i < len(elements) - 1 else ""
            next_context = next_item.text[:300] if hasattr(next_item, 'text') else "None"
            
            table_context = f"Context Before: {prev_context}\nContext After: {next_context}"
            
            # Summarize the table for semantic search (retrieve this)
            print(f"Summarizing table on page {metadata['page_number']}...")
            summary = summarize_table(table_markdown, table_context, llm)
            summary_doc = Document(
                page_content=summary,
                metadata={**metadata, "type": "table_summary", "original_table": table_markdown}
            )
            table_summaries.append(summary_doc)
            
            # Raw Markdown doc (use this in context)
            raw_doc = Document(
                page_content=table_markdown,
                metadata={**metadata, "type": "table", "table_context": table_context}
            )
            table_raw_docs.append(raw_doc)
            
        # For non-table elements, check if they have text content and create text docs
        elif hasattr(item, 'text'):
            if len(item.text.strip()) > 20:
                text_doc = Document(
                    page_content=item.text,
                    metadata={**metadata, "type": "text"}
                )
                text_docs.append(text_doc)
            
    print(f"Finished processing {pdf}. Total text elements: {len(text_docs)}. Table elements: {len(table_raw_docs)}.\n")

Checking: c:\Users\Marcus\Desktop\Python Projs\VAA\Virtual-Academic-Advisor-with-RAG\old_notebook_files\RAG\ProgramBooklet\BEngBSc_Scheme_IAIE_46409_2526.pdf, exists: True
PDF: BEngBSc_Scheme_IAIE_46409_2526.pdf -> Programme: BEng/BSc (Hons) Scheme in Information and Artificial Intelligence Engineering | BEng/BSc (Hons) in IAIE

Converting BEngBSc_Scheme_IAIE_46409_2526.pdf into markdown.



[INFO] 2026-05-10 10:38:08,856 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-10 10:38:08,866 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Marcus\anaconda3\envs\FYPEnv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-10 10:38:08,866 [RapidOCR] main.py:53: Using C:\Users\Marcus\anaconda3\envs\FYPEnv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-05-10 10:38:08,990 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-10 10:38:08,994 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\Marcus\anaconda3\envs\FYPEnv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-05-10 10:38:08,994 [RapidOCR] main.py:53: Using C:\Users\Marcus\anaconda3\envs\FYPEnv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-05-10 10:38:09,076 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-10 10:38:09,093 [R

Summarizing table on page 3...


Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.


Summarizing table on page 4...


...

Summarizing table on page 41...


Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.


Summarizing table on page 42...


Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.


Summarizing table on page 43...
Finished processing PhDMPhil_EEE_46601_2526.pdf. Total text elements: 11377. Table elements: 1176.



In [5]:
# Print an example of the extracted text and table summary documents

print("Example Text Document:")
if text_docs:
    print(text_docs[12].page_content)  # Print the characters of the first text document 

Example Text Document:
Students  will  be  awarded  one  of  the  following  awards  upon  successful  completion  of  the graduation requirements of the programme:
- Bachelor of Engineering (Honours) in Electronic Systems and Internet-of-Things 電子系統及物聯網 ( 榮譽 ) 工學士學位
- Bachelor of Science (Honours) in Artificial Intelligence and Information Engineering 人工智能及資訊工程學 ( 榮譽 ) 理學士學位
- Bachelor of Science (Honours) in Information Security 資訊安全 ( 榮譽 ) 理學士學位
Students admitted to the Scheme complete a common curriculum in Year 1 and then complete their preferred award in the next three years until graduation.


In [6]:
print("\nExample Table Summary Document:")
if table_summaries:
    print(table_summaries[27].page_content)  # Print the characters of the first table summary document


Example Table Summary Document:
**Concise Summary:**

The table outlines the progression pattern of the BSc (Hons) in Artificial Intelligence and Information Engineering program for Normal Year 1 Intake. The key information used for vector search in the RAG system is:

* **Year 1:** Students take a mix of core, cluster-area requirement (CAR), free elective, and training credits.
	+ Core modules: Basic Mathematics I & II, English LCR Subject 1 & 2, Information Technology, Introduction to Innovation and Entrepreneurship, and Applied Engineering Fundamentals.
	+ CAR modules: Cluster-Area Requirement subject 1.
* **Year 2:** Students continue with core, CAR, free elective, and training credits.
	+ Core modules: Data Structures and Algorithms, Foundations of Data Science, Digital and Computer Systems, Foundation Techniques in Artificial Intelligence, and Database System.
	+ CAR modules: Cluster-Area Requirement subject 2.
* **Year 3 & 4:** Students take a mix of core, technical elective, a

### 2. Text Splitting

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(text_docs)

all_chunks = [copy.deepcopy(chunk) for chunk in chunks] + [copy.deepcopy(doc) for doc in table_summaries]

for i, chunk in enumerate(all_chunks):
    source = chunk.metadata.get("source", "N/A")
    page = chunk.metadata.get("page_number", "N/A")
    # Handle None page_number for chunk_id
    page_str = str(page) if page is not None else "0"
    programme = chunk.metadata.get("programme_title", "N/A")
    heading = chunk.metadata.get("headings", [])
    
    # Identify if it's a summary or a standard text chunk
    is_table = chunk.metadata.get("type") == "table_summary"
    chunk_type = "table_summary" if is_table else "text_chunk"
    
    chunk.metadata["chunk_id"] = f"PolyU_{source}_p{page_str}_{chunk_type}_{i}"

    # Header for the LLM to understand the context of the retrieved snippet
    chunk.metadata["source"] = eee_path
    chunk.page_content = f"--- Programme booklet: {programme}: {heading} ---\n\n{chunk.page_content}"

    # Remove problematic keys or None values
    keys_to_delete = ["programme_title", "coordinates", "languages"]
    for key in keys_to_delete:
        if key in chunk.metadata:
            del chunk.metadata[key]
            
    # CRITICAL: ChromaDB does not allow None values in metadata
    for key, value in list(chunk.metadata.items()):
        if value is None:
            if key == "page_number":
                chunk.metadata[key] = 0
            elif key == "headings":
                chunk.metadata[key] = ""
            else:
                chunk.metadata[key] = "N/A"
        elif isinstance(value, list):
            # ChromaDB also doesn't like lists in metadata values
            chunk.metadata[key] = ", ".join(map(str, value))
        
print("Number of chunks: ", len(all_chunks))

long_chunks = [c for c in all_chunks if len(c.page_content) > 1500]
print(f"Chunks over 1500 chars: {len(long_chunks)}")
if long_chunks:
    print(f"Longest chunk: {len(long_chunks[0].page_content)} chars")


Number of chunks:  14876
Chunks over 1500 chars: 4
Longest chunk: 1608 chars


In [8]:
# Inspect metadata for None values in all_chunks
none_metadata_keys = set()
for chunk in all_chunks:
    for key, value in chunk.metadata.items():
        if value is None:
            none_metadata_keys.add(key)

print(f"Metadata keys with None values: {none_metadata_keys}")
if all_chunks:
    print(f"Example metadata: {all_chunks[0].metadata}")


Metadata keys with None values: set()
Example metadata: {'source': 'https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/', 'content_type': 'pdf', 'page_number': 0, 'headings': '', 'type': 'text', 'chunk_id': 'PolyU_BEngBSc_Scheme_IAIE_46409_2526.pdf_p0_text_chunk_0'}


#### Additional Step: Saving all components for any later use.

In [ ]:
file_name = "table_docs_alt.pkl"

data_to_store = {
    "text_chunks": chunks,
    "table_summaries": table_summaries,
    "table_raw_docs": table_raw_docs
}

with open(file_name, 'wb') as f:
    pickle.dump(data_to_store, f)
print(f"Exported processing results to {file_name}")

Exported processing results to table_docs_alt.pkl


### 3. Document Embedding in Chroma

In [10]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

In [11]:
client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

1094

In [12]:
import requests
print(requests.get("http://127.0.0.1:11434/api/tags", timeout=5).status_code)

200


In [13]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, all_chunks))

In [14]:
for i, chunk in enumerate(all_chunks):
    # print(f"Adding chunk {i+1}/{len(all_chunks)} to ChromaDB. Metadata: {chunk.metadata}\n")
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i+3000)]
    )

print(f"Added {len(all_chunks)} chunks into ChromaDB")


Added 14876 chunks into ChromaDB


### 4. Simple Testing

In [15]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "What courses are offered in the first year of IAIE programme?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    '''
    if "original_table" in result.metadata:
        print("[Swapping for Raw Markdown Table]")
        new_result = result.metadata["original_table"]
    else:
        new_result = result.page_content
    '''

    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

C:\Users\Marcus\AppData\Local\Temp\ipykernel_29272\4165432853.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


Content: --- Programme booklet: BEng/BSc (Hons) Scheme in Information and Artificial Intelligence Engineering | BEng/BSc (Hons) in IAIE: ['BSc (Hons) in Information Security (Normal Year 1 Intake)'] ---

Year 1 (31 academic credits + 2 training credits).Semester 1 (13 or 16 credits + 1 training credit) = Information Technology (3)^. ENG2003, Year 1 (31 academic credits + 2 training credits).Semester 2 (15 or 18 credits + 1 training credit) = EIE1005. ENG2003, Year 1 (31 academic credits + 2 training credits).Semester 2 (15 or 18 credits + 1 training credit) = Fundamental AI and Data Analytics (2). MM1031, Year 1 (31 academic credits + 2 training credits).Semester 1 (13 or 16 credits + 1 training credit) = Introduction to Innovation and Entrepreneurship (1). MM1031, Year 1 (31 academic credits + 2 training credits).Semester 2 (15 or 18 credits + 1 training credit) = ELCXXXX. MM1031, Year 1 (31 academic credits + 2 training credits).Semester 2 (15 or 18 credits + 1 training credit) = Eng